# 第十八课｜高级可扩展接口（Advanced eXtensible Interface, AXI）只学我们需要的部分

上一课知道了 burst。在 **片上系统（System on Chip, SoC）** 与 **现场可编程门阵列（Field-Programmable Gate Array, FPGA）** 平台中，模块还需要一套约定表达地址、数据与“现在能不能接收”。

今天只解决一个核心问题：

> **发送方有有效数据、接收方也准备好时，怎样确认一次传输真正发生？**

主要新概念：**AXI 的 transaction/beat 与 VALID/READY 握手。**

## 1. 概念账本

**已经知道：** host/可编程逻辑（PL）、外部 DDR（Double Data Rate Synchronous Dynamic Random-Access Memory）、burst、backpressure。

**今天学习：**
- **高级可扩展接口（Advanced eXtensible Interface, AXI）**：ARM/FPGA 系统常见的一组片上通信协议；
- **transaction**：一次较完整的读/写操作；
- **beat**：transaction 中的一次数据传输单位；
- **VALID/READY handshake**。

**只预告：** 多 channel、ordering、outstanding transaction、cache/coherency 等高级内容。

## 2. 为什么这和 FIFO backpressure 很像？

sender 用 VALID 表示“当前 payload 有效”，receiver 用 READY 表示“现在能接收”。

**只有在同一个有效 clock edge 上 VALID=1 且 READY=1，transfer 才发生。**

## 3. 最小握手图

<div style="max-width:760px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 340" role="img" aria-label="VALID READY handshake diagram" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="axi-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L0,6 L9,3 z" fill="#333333"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="24" text-anchor="middle">
    <rect x="40" y="30" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="165" y="75" fill="#222222">sender VALID payload</text>
    <rect x="470" y="30" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="595" y="75" fill="#222222">receiver READY</text>
    <rect x="255" y="150" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="195" fill="#222222">handshake check</text>
    <rect x="255" y="255" width="250" height="72" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="300" fill="#222222">accepted transfer</text>
  </g>
  <path d="M165 102 C165 128,250 128,320 150" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#axi-arrow)"/>
  <path d="M595 102 C595 128,510 128,440 150" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#axi-arrow)"/>
  <path d="M380 222 L380 255" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#axi-arrow)"/>
</svg>
</div>

VALID=1、READY=0 时，sender 必须保留尚未接受的 payload。\n\n更严格地说：**一旦 sender 拉高 VALID，在 handshake 完成前 VALID 不能自行撤掉；与这次传输相关的 payload/control 也必须保持稳定，直到某个有效 clock edge 上 VALID=1 且 READY=1。**

## 4. Run：哪些 cycle 真正发生 transfer？

cycle 1 发生 stall，所以 data=11 还没有被接受；cycle 2 再呈现同一 payload。

In [ ]:
valid = [True, True, True, False, True]
ready = [True, False, True, True, True]
data  = [10, 11, 11, 0, 12]

accepted_cycles = []
accepted_data = []

for cycle, (v, r, d) in enumerate(zip(valid, ready, data)):
    if v and r:
        accepted_cycles.append(cycle)
        accepted_data.append(d)

print("accepted cycles:", accepted_cycles)
print("accepted data:", accepted_data)


## 5. Observe

transfer 发生在 cycle 0、2、4，对应数据 10、11、12。cycle 1 是等待，不是“失败的 transaction”。

## 6. transaction、burst、beat 的关系

<div style="max-width:760px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 330" role="img" aria-label="transaction burst beat diagram" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="burst-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L0,6 L9,3 z" fill="#333333"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="24" text-anchor="middle">
    <rect x="210" y="25" width="340" height="70" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="69" fill="#222222">read/write transaction</text>
    <rect x="185" y="130" width="390" height="70" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="174" fill="#222222">burst of one or more beats</text>
    <rect x="150" y="235" width="460" height="70" rx="4" fill="#eef0ff" stroke="#6f63ff" stroke-width="2"/>
    <text x="380" y="279" fill="#222222">each beat advances on VALID and READY</text>
  </g>
  <path d="M380 95 L380 130" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#burst-arrow)"/>
  <path d="M380 200 L380 235" fill="none" stroke="#333333" stroke-width="2.5" marker-end="url(#burst-arrow)"/>
</svg>
</div>

一个 burst transaction 可以包含多个 data beat，每个 beat 的推进仍受 handshake 控制。

## 7. 为什么只学 AXI subset？

完整 AXI 很大。本项目先掌握 valid/ready、read/write transaction 的概念、contiguous burst，以及用平台 IP/controller 接 DDR。只有后续需要时才引入更复杂规则。

## 8. 把协议细节藏在存储后端

如果以后把 synapse store 从片上存储换成 DDR，上层 synapse engine 不应该因此改写自己的业务语义。

更稳健的边界是：存储后端负责 AXI/DDR 等协议细节；synapse engine 继续消费同一种 **source → synapse record 数据流**。这样更换存储实现时，上层计算逻辑不需要一起变化。

## 9. Try It

把 cycle 2 的 READY 改成 `False`，再让 cycle 3 成为 VALID=True、READY=True、data=11。预测 accepted cycle 如何变化。

## 10. 作业

[第 18 课作业：找出 VALID/READY 真正接受的 data beat](../../exercises/zh/18_axi_subset.ipynb)

## 11. AI Task

让 AI 逐 cycle 标记 accepted / stalled / idle。检查它是否错误地把 VALID 单独为 1 当成 transfer。

## 12. Human Check

解释 AXI 为什么是一组协议；VALID 与 READY 各自表达什么；为什么 VALID=1/READY=0 时 payload 不能被当成已消费；为什么把 DDR/AXI 藏在稳定的 synapse-stream 边界后面能减少耦合。

## 13. Engineering Handoff

对应 `RMD-014A / RMD-015 / RMD-016`：比较 random/small transaction 与 burst 的有效 bandwidth；随后把 synapse store 换到 DDR 而保持上层 stream 语义，并建立 throughput baseline。

## 14. 项目追踪 Project Trace

- Lesson: `LSN-018`
- Mapping: `RMD-014A / RMD-015 / RMD-016`
- Interface focus: transaction / beat / VALID-READY
- Architecture boundary: DDR backend hidden behind `IF-SYNAPSE-STREAM`

## 15. Exit Ticket

给你一段 VALID/READY trace，你能逐 cycle 判断 transfer 是否发生，并说明 DDR/AXI 为什么不应泄漏成 synapse engine 的业务语义。